In [ ]:
# 1. Crop hotstart
# 2. resample to 500m grid

In [1]:
import xarray as xr
import numpy as np

from scipy.interpolate import RegularGridInterpolator

In [3]:
hotstart_200m = xr.open_dataset('/export/lv1/user/jvandermolen/model_output/active_runs/boundaries/dws_200m_nwes/restart_201501_dws200m_bio.nc')
bathy_200m = xr.open_dataset('/export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_smoothing/topo_adjusted_dws_200m_2009.nc')

print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
print("Bathy 200m dimensions:", dict(bathy_200m.dims))

# Scale xax and yax by 200 and assign back
hotstart_200m = hotstart_200m.assign_coords(
    xax=hotstart_200m.xax * 200,
    yax=hotstart_200m.yax * 200,
)

print("Verifying scaled offsets:")
print(f"Hotstart xax (x200): {hotstart_200m.xax.values}")
print(f"Hotstart yax (x200): {hotstart_200m.yax.values}")


Hotstart 200m dimensions: {'xax': 820, 'yax': 496, 'zax': 31}
Bathy 200m dimensions: {'yc': 486, 'xc': 820, 'xx': 821, 'yx': 487}
Verifying scaled offsets:
Hotstart xax (x200): [     0.    200.    400.    600.    800.   1000.   1200.   1400.   1600.
   1800.   2000.   2200.   2400.   2600.   2800.   3000.   3200.   3400.
   3600.   3800.   4000.   4200.   4400.   4600.   4800.   5000.   5200.
   5400.   5600.   5800.   6000.   6200.   6400.   6600.   6800.   7000.
   7200.   7400.   7600.   7800.   8000.   8200.   8400.   8600.   8800.
   9000.   9200.   9400.   9600.   9800.  10000.  10200.  10400.  10600.
  10800.  11000.  11200.  11400.  11600.  11800.  12000.  12200.  12400.
  12600.  12800.  13000.  13200.  13400.  13600.  13800.  14000.  14200.
  14400.  14600.  14800.  15000.  15200.  15400.  15600.  15800.  16000.
  16200.  16400.  16600.  16800.  17000.  17200.  17400.  17600.  17800.
  18000.  18200.  18400.  18600.  18800.  19000.  19200.  19400.  19600.
  19800.  20000.  20

/tmp/ipykernel_94530/1712769170.py:2: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  bathy_200m = xr.open_dataset('/export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_smoothing/topo_adjusted_dws_200m_2009.nc')
/tmp/ipykernel_94530/1712769170.py:4: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension 

In [4]:
print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
print("Bathy 200m dimensions:", dict(bathy_200m.dims))

Hotstart 200m dimensions: {'zax': 31, 'yax': 496, 'xax': 820}
Bathy 200m dimensions: {'yc': 486, 'xc': 820, 'xx': 821, 'yx': 487}


/tmp/ipykernel_94530/3480680765.py:1: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
/tmp/ipykernel_94530/3480680765.py:2: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Bathy 200m dimensions:", dict(bathy_200m.dims))


In [5]:
# Verify the offset between hotstart and bathymetry grids
print("Verifying offsets:")
print(f"Hotstart xax[0]: {hotstart_200m.xax.values[0]}")
print(f"Bathy xc[0]: {bathy_200m.xc.values[0]}")
print(f"Hotstart yax[10]: {hotstart_200m.yax.values[10]}")
print(f"Bathy yc[0]: {bathy_200m.yc.values[0]}")

print(f"Hotstart yax[10]: {hotstart_200m.yax.values[0:11]}")
print(f"Bathy yc[0]: {bathy_200m.yc.values[0:11]}")

# Check if they match as expected
x_offset = 0
y_offset = -10
print(f"\nExpected x offset: {x_offset} (hotstart starts at same x as bathy)")
print(f"Expected y offset: {y_offset} (hotstart starts 10 cells before bathy in y)")


# Crop the hotstart to match bathymetry dimensions
# Bathymetry: xc=820, yc=486
# Hotstart: xax=820, yax=496
# Slice xax[0:820], yax[10:496] (10 + 486 = 496)
cropped_hotstart = hotstart_200m.isel(xax=slice(0, 820), yax=slice(10, 496))

print("\nCropped hotstart dimensions:", dict(cropped_hotstart.dims))

# Verify that coordinates now match
print("\nVerifying cropped coordinates match bathymetry:")
print(f"Cropped hotstart xax[0]: {cropped_hotstart.xax.values[0]}")
print(f"Bathy xc[0]: {bathy_200m.xc.values[0]}")
print(f"Cropped hotstart yax[0]: {cropped_hotstart.yax.values[0]}")
print(f"Bathy yc[0]: {bathy_200m.yc.values[0]}")

# Check if all spatial dimensions match
print(f"\nSpatial dimensions match: xax={cropped_hotstart.dims['xax']} == xc={bathy_200m.dims['xc']}, yax={cropped_hotstart.dims['yax']} == yc={bathy_200m.dims['yc']}")


Verifying offsets:
Hotstart xax[0]: 0.0
Bathy xc[0]: 0.0
Hotstart yax[10]: 0.0
Bathy yc[0]: 0.0
Hotstart yax[10]: [200. 200. 200. 200. 200. 200. 200. 200. 200. 200.   0.]
Bathy yc[0]: [   0.  200.  400.  600.  800. 1000. 1200. 1400. 1600. 1800. 2000.]

Expected x offset: 0 (hotstart starts at same x as bathy)
Expected y offset: -10 (hotstart starts 10 cells before bathy in y)

Cropped hotstart dimensions: {'zax': 31, 'yax': 486, 'xax': 820}

Verifying cropped coordinates match bathymetry:
Cropped hotstart xax[0]: 0.0
Bathy xc[0]: 0.0
Cropped hotstart yax[0]: 0.0
Bathy yc[0]: 0.0

Spatial dimensions match: xax=820 == xc=820, yax=486 == yc=486


/tmp/ipykernel_94530/3369298884.py:24: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("\nCropped hotstart dimensions:", dict(cropped_hotstart.dims))
/tmp/ipykernel_94530/3369298884.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"\nSpatial dimensions match: xax={cropped_hotstart.dims['xax']} == xc={bathy_200m.dims['xc']}, yax={cropped_hotstart.dims['yax']} == yc={bathy_200m.dims['yc']}")


In [ ]:
# Save cropped hotstart
#cropped_hotstart.to_netcdf('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc')
#print("Cropped hotstart saved as '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc'")

Cropped hotstart saved as '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc'


In [6]:
# ncmerge -v /export/lv9/user/cgiannopoulos/model_output/active_runs/dws_500m/dws_500m/26x20/2015_backup/01/restart.00??.in ./restart_201501.in

hydro_500m = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_hydro_201501.in')
# hotstart_200m = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc')

hotstart_200m = cropped_hotstart
# Print dimensions to confirm
print("Hydro 500m dimensions:", dict(hydro_500m.dims))
print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))

# Get target coordinates for 500m grid
xc_500 = hydro_500m.xax.values
yc_500 = hydro_500m.yax.values

print(f"Target grid sizes: xc={len(xc_500)}, yc={len(yc_500)}")

# Print coordinate ranges for debugging
source_x = hotstart_200m.xax.values
source_y = hotstart_200m.yax.values
print(f"Source x range: {source_x.min():.2f} to {source_x.max():.2f}")
print(f"Source y range: {source_y.min():.2f} to {source_y.max():.2f}")
print(f"Target x range: {xc_500.min():.2f} to {xc_500.max():.2f}")
print(f"Target y range: {yc_500.min():.2f} to {yc_500.max():.2f}")

Hydro 500m dimensions: {'xax': 338, 'yax': 200, 'zax': 11}
Hotstart 200m dimensions: {'zax': 31, 'yax': 486, 'xax': 820}
Target grid sizes: xc=338, yc=200
Source x range: 0.00 to 163800.00
Source y range: 0.00 to 97000.00
Target x range: 0.00 to 168500.00
Target y range: -3000.00 to 96500.00


/tmp/ipykernel_94530/3249642120.py:8: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Hydro 500m dimensions:", dict(hydro_500m.dims))
/tmp/ipykernel_94530/3249642120.py:9: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))


In [7]:
# Crop hydro_500m to the requested coordinate window before interpolation
x_min, x_max = 0.0, 163800.0
y_min, y_max = 0.0, 96500.0

x_mask = (hydro_500m.xax >= x_min) & (hydro_500m.xax <= x_max)
y_mask = (hydro_500m.yax >= y_min) & (hydro_500m.yax <= y_max)

hydro_500m = hydro_500m.isel(xax=x_mask, yax=y_mask)

# Update target coordinates after cropping
xc_500 = hydro_500m.xax.values
yc_500 = hydro_500m.yax.values

print("Cropped Hydro 500m dimensions:", dict(hydro_500m.dims))
print(f"Cropped target x range: {xc_500.min():.2f} to {xc_500.max():.2f}")
print(f"Cropped target y range: {yc_500.min():.2f} to {yc_500.max():.2f}")



Cropped Hydro 500m dimensions: {'xax': 328, 'yax': 194, 'zax': 11}
Cropped target x range: 0.00 to 163500.00
Cropped target y range: 0.00 to 96500.00


/tmp/ipykernel_94530/78211015.py:14: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Cropped Hydro 500m dimensions:", dict(hydro_500m.dims))


In [8]:
# Create landmask from bathymetry missing values
# Assuming missing_value = -10 as in the 200m bathy
bathy_500m = xr.open_dataset('/export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_resampling/topo_dws_500m.nc')

missing_value = bathy_500m.bathymetry.missing_value if 'missing_value' in bathy_500m.bathymetry.attrs else -10.0
water_mask = bathy_500m.bathymetry.values > missing_value  # Water where bathymetry > missing_value
print(f"Water cells: {np.sum(water_mask)} out of {water_mask.size}")


Water cells: 31728 out of 63632


In [9]:
# Create meshgrid for interpolation points
X_grid, Y_grid = np.meshgrid(xc_500, yc_500, indexing='ij')

# Create new dataset for 500m hotstart
hotstart_500m = xr.Dataset()

# Set coordinates (keep names xax, yax, zax)
hotstart_500m['xax'] = ('xax', xc_500)
hotstart_500m['yax'] = ('yax', yc_500)
hotstart_500m['zax'] = hotstart_200m.zax  # Copy zax unchanged

# Copy global attributes
hotstart_500m.attrs = hotstart_200m.attrs

In [10]:
hotstart_200m.data_vars

Data variables:
    loop          int32 4B ...
    julianday     int32 4B ...
    secondsofday  int32 4B ...
    timestep      float64 8B ...
    z             (yax, xax) float64 3MB ...
    zo            (yax, xax) float64 3MB ...
    U             (yax, xax) float64 3MB ...
    SlUx          (yax, xax) float64 3MB ...
    Slru          (yax, xax) float64 3MB ...
    V             (yax, xax) float64 3MB ...
    SlVx          (yax, xax) float64 3MB ...
    Slrv          (yax, xax) float64 3MB ...
    ssen          (yax, xax) float64 3MB ...
    ssun          (yax, xax) float64 3MB ...
    ssvn          (yax, xax) float64 3MB ...
    sseo          (yax, xax) float64 3MB ...
    ssuo          (yax, xax) float64 3MB ...
    ssvo          (yax, xax) float64 3MB ...
    Uint          (yax, xax) float64 3MB ...
    Vint          (yax, xax) float64 3MB ...
    Uadv          (yax, xax) float64 3MB ...
    Vadv          (yax, xax) float64 3MB ...
    uu            (zax, yax, xax) float64 99MB .

In [ ]:
# Interpolate each variable only at water cells
for var_name in hotstart_200m.data_vars:
    var = hotstart_200m[var_name]
    print(f"Interpolating {var_name} with dims {var.dims}")

    if var.dims == ('xax', 'yax'):
        # 2D variable interpolation
        data = var.values
        interp_func = RegularGridInterpolator((source_x, source_y), data, method='linear', bounds_error=False, fill_value=None)
        # Only interpolate at water points
        water_indices = np.where(water_mask.ravel())[0]
        target_x = X_grid.ravel()[water_indices]
        target_y = Y_grid.ravel()[water_indices]
        target_points = np.column_stack((target_x, target_y))
        interp_values = interp_func(target_points)
        new_values = np.full((len(xc_500), len(yc_500)), 0.0)
        new_values.ravel()[water_indices] = interp_values
        hotstart_500m[var_name] = (('xax', 'yax'), new_values)

    elif var.dims == ('xax', 'yax', 'zax'):
        # 3D variable interpolation (interpolate each z-level)
        new_values = np.zeros((31, len(xc_500), len(yc_500)))
        for k in range(31):
            data_k = var.values[k, :, :]
            interp_func = RegularGridInterpolator((source_x, source_y), data_k, method='linear', bounds_error=False, fill_value=None)
            # Only interpolate at water points
            water_indices = np.where(water_mask.ravel())[0]
            target_x = X_grid.ravel()[water_indices]
            target_y = Y_grid.ravel()[water_indices]
            target_points = np.column_stack((target_x, target_y))
            interp_values = interp_func(target_points)
            new_values[k].ravel()[water_indices] = interp_values
        hotstart_500m[var_name] = (('xax', 'yax', 'zax'), new_values)

    else:
        # Copy variables with other dimensions unchanged (scalars, 1D, etc.)
        hotstart_500m[var_name] = var

print("Interpolation complete. New dimensions:", dict(hotstart_500m.dims))

# Check for any NaN values in the interpolated data
nan_count = 0
for var_name in hotstart_500m.data_vars:
    var = hotstart_500m[var_name]
    if np.any(np.isnan(var.values)):
        nan_count += np.sum(np.isnan(var.values))
        print(f"Warning: {var_name} contains {np.sum(np.isnan(var.values))} NaN values")

if nan_count == 0:
    print("No NaN values found in the interpolated data.")
else:
    print(f"Total NaN values: {nan_count}")

In [11]:
hydro_var_names = set(hydro_500m.data_vars)
print(hydro_var_names)

{'S', 'zo', 'SlVx', 'U', 'uuEx', 'SlUx', 'ho', 'Slrv', 'ww', 'Slru', 'eps', 'timestep', 'vv', 'hn', 'sseo', 'T', 'ssvo', 'uu', 'julianday', 'Uadv', 'ssen', 'Vint', 'ssvn', 'tke', 'Vadv', 'z', 'secondsofday', 'nuh', 'ssun', 'vvEx', 'num', 'loop', 'ssuo', 'Uint', 'V'}


In [12]:
# Interpolate biological variables only
# Hydrological variables (present in hydro_500m) will be replaced directly from the hydro file
hydro_var_names = set(hydro_500m.data_vars)

for var_name in hotstart_200m.data_vars:
    if var_name in hydro_var_names:
        print(f"Skipping {var_name} (hydrological variable, will be taken from hydro file)")
        continue

    var = hotstart_200m[var_name]
    print(f"Interpolating {var_name} with dims {var.dims}")

    if var.dims == ('xax', 'yax'):
        # 2D variable interpolation
        data = var.values
        interp_func = RegularGridInterpolator((source_x, source_y), data, method='linear', bounds_error=False, fill_value=None)
        # Only interpolate at water points
        water_indices = np.where(water_mask.ravel())[0]
        target_x = X_grid.ravel()[water_indices]
        target_y = Y_grid.ravel()[water_indices]
        target_points = np.column_stack((target_x, target_y))
        interp_values = interp_func(target_points)
        new_values = np.full((len(xc_500), len(yc_500)), 0.0)
        new_values.ravel()[water_indices] = interp_values
        hotstart_500m[var_name] = (('xax', 'yax'), new_values)

    elif var.dims == ('xax', 'yax', 'zax'):
        # 3D variable interpolation (interpolate each z-level)
        new_values = np.zeros((31, len(xc_500), len(yc_500)))
        for k in range(31):
            data_k = var.values[k, :, :]
            interp_func = RegularGridInterpolator((source_x, source_y), data_k, method='linear', bounds_error=False, fill_value=None)
            # Only interpolate at water points
            water_indices = np.where(water_mask.ravel())[0]
            target_x = X_grid.ravel()[water_indices]
            target_y = Y_grid.ravel()[water_indices]
            target_points = np.column_stack((target_x, target_y))
            interp_values = interp_func(target_points)
            new_values[k].ravel()[water_indices] = interp_values
        hotstart_500m[var_name] = (('xax', 'yax', 'zax'), new_values)

    else:
        # Copy variables with other dimensions unchanged (scalars, 1D, etc.)
        hotstart_500m[var_name] = var

print("Interpolation complete. New dimensions:", dict(hotstart_500m.dims))

# Check for any NaN values in the interpolated data
nan_count = 0
for var_name in hotstart_500m.data_vars:
    var = hotstart_500m[var_name]
    if np.any(np.isnan(var.values)):
        nan_count += np.sum(np.isnan(var.values))
        print(f"Warning: {var_name} contains {np.sum(np.isnan(var.values))} NaN values")

if nan_count == 0:
    print("No NaN values found in the interpolated data.")
else:
    print(f"Total NaN values: {nan_count}")

Skipping loop (hydrological variable, will be taken from hydro file)
Skipping julianday (hydrological variable, will be taken from hydro file)
Skipping secondsofday (hydrological variable, will be taken from hydro file)
Skipping timestep (hydrological variable, will be taken from hydro file)
Skipping z (hydrological variable, will be taken from hydro file)
Skipping zo (hydrological variable, will be taken from hydro file)
Skipping U (hydrological variable, will be taken from hydro file)
Skipping SlUx (hydrological variable, will be taken from hydro file)
Skipping Slru (hydrological variable, will be taken from hydro file)
Skipping V (hydrological variable, will be taken from hydro file)
Skipping SlVx (hydrological variable, will be taken from hydro file)
Skipping Slrv (hydrological variable, will be taken from hydro file)
Skipping ssen (hydrological variable, will be taken from hydro file)
Skipping ssun (hydrological variable, will be taken from hydro file)
Skipping ssvn (hydrological 

/tmp/ipykernel_94530/2494548822.py:46: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Interpolation complete. New dimensions:", dict(hotstart_500m.dims))


Interpolation complete. New dimensions: {'xax': 328, 'yax': 194, 'zax': 31}
Total NaN values: 102209511


In [14]:
nlayers = len(hotstart_500m.zax) - 1
print(f"Number of layers (nlayers): {nlayers}")

nlayers_new = 10
print(f"New number of layers (nlayers_new): {nlayers_new}")

layer_ratio = nlayers / nlayers_new
print(f"Layer ratio (nlayers / nlayers_new): {layer_ratio:.2f} \n has to be integer for proper interpolation")

Number of layers (nlayers): 30
New number of layers (nlayers_new): 10
Layer ratio (nlayers / nlayers_new): 3.00 
 has to be integer for proper interpolation


In [15]:
# Subsample in z-direction for 10 layers (11 levels)
# Original zax: 31 levels (0 to 30)
# For 10 layers: subsample every 3 levels -> 11 levels: 0,3,6,...,30
z_step = 3
new_zax_indices = np.arange(0, 31, z_step)  # [0,3,6,...,30]
print(f"Subsampling zax from {len(hotstart_500m.zax)} to {len(new_zax_indices)} levels")

# Create subsampled dataset
hotstart_500m_10layer = hotstart_500m.isel(zax=new_zax_indices)

# Update zax coordinate to new values from 0 to 10
new_zax_values = np.arange(len(new_zax_indices), dtype=float)  # 0, 1, 2, ..., 10
hotstart_500m_10layer = hotstart_500m_10layer.assign_coords(zax=new_zax_values)

print("Z-subsampling complete. New dimensions:", dict(hotstart_500m_10layer.dims))
print(f"New zax values: {new_zax_values}")

# Check for any NaN values in the subsampled data
nan_count = 0
for var_name in hotstart_500m_10layer.data_vars:
    var = hotstart_500m_10layer[var_name]
    if np.any(np.isnan(var.values)):
        nan_count += np.sum(np.isnan(var.values))
        print(f"Warning: {var_name} contains {np.sum(np.isnan(var.values))} NaN values")

if nan_count == 0:
    print("No NaN values found in the subsampled data.")
else:
    print(f"Total NaN values: {nan_count}")



Subsampling zax from 31 to 11 levels
Z-subsampling complete. New dimensions: {'xax': 328, 'yax': 194, 'zax': 11}
New zax values: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]
Total NaN values: 39894951


/tmp/ipykernel_94530/2411734415.py:15: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Z-subsampling complete. New dimensions:", dict(hotstart_500m_10layer.dims))


In [17]:
# Replace hydrological variables in hotstart_500m_10layer with values from hydro_500m
# hydro_500m is already on the 500m grid; biological variables were interpolated in the previous cell

for var_name in hydro_500m.data_vars:
    if hydro_500m[var_name].dims == hotstart_500m_10layer[var_name].dims if var_name in hotstart_500m_10layer.data_vars else False:
        print(f"Replacing {var_name} (dims match: {hydro_500m[var_name].dims})")
        hotstart_500m_10layer[var_name] = hydro_500m[var_name]
    else:
        # Variable not yet in hotstart or dims differ — assign directly
        try:
            hotstart_500m_10layer[var_name] = hydro_500m[var_name]
            print(f"Adding {var_name} from hydro file (dims: {hydro_500m[var_name].dims})")
        except Exception as e:
            print(f"Could not add {var_name}: {e}")

print("\nHydrological variable replacement complete.")

Adding loop from hydro file (dims: ())
Adding julianday from hydro file (dims: ())
Adding secondsofday from hydro file (dims: ())
Adding timestep from hydro file (dims: ())
Adding z from hydro file (dims: ('yax', 'xax'))
Adding zo from hydro file (dims: ('yax', 'xax'))
Adding U from hydro file (dims: ('yax', 'xax'))
Adding SlUx from hydro file (dims: ('yax', 'xax'))
Adding Slru from hydro file (dims: ('yax', 'xax'))
Adding V from hydro file (dims: ('yax', 'xax'))
Adding SlVx from hydro file (dims: ('yax', 'xax'))
Adding Slrv from hydro file (dims: ('yax', 'xax'))
Adding ssen from hydro file (dims: ('yax', 'xax'))
Adding ssun from hydro file (dims: ('yax', 'xax'))
Adding ssvn from hydro file (dims: ('yax', 'xax'))
Adding sseo from hydro file (dims: ('yax', 'xax'))
Adding ssuo from hydro file (dims: ('yax', 'xax'))
Adding ssvo from hydro file (dims: ('yax', 'xax'))
Adding Uint from hydro file (dims: ('yax', 'xax'))
Adding Vint from hydro file (dims: ('yax', 'xax'))
Adding Uadv from hydro

In [ ]:
# Save the resampled hotstart file
#hotstart_500m.to_netcdf('../restart_dws_500m_201412_hydro.nc')

hotstart_500m_10layer.to_netcdf('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_2015_01.in')

print("Saved to '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_2015_01.in'")

In [ ]:
# ncmerge -v /export/lv9/user/cgiannopoulos/model_output/active_runs/dws_500m/dws_500m/26x20/2015_backup/01/restart.00??.in ./restart_201501.in

hydro_500m_origin = xr.open_dataset('/export/lv9/user/qzhan/model_output/active_runs/dws_500m/dws_500m/26x20/2015/01/restart_201501.in')

# Print dimensions to confirm
print("Hydro 500m dimensions:", dict(hydro_500m_origin.dims))
print("Hotstart 500m dimensions:", dict(hotstart_500m_10layer.dims))

# Get target coordinates for 500m grid
xc_500_target = hydro_500m_origin.xax.values
yc_500_target = hydro_500m_origin.yax.values

print(f"Target x range: {xc_500_target.min():.2f} to {xc_500_target.max():.2f}")
print(f"Target y range: {yc_500_target.min():.2f} to {yc_500_target.max():.2f}")

# Print coordinate ranges for debugging
current_x = hotstart_500m_10layer.xax.values
current_y = hotstart_500m_10layer.yax.values
print(f"current x range: {current_x.min():.2f} to {current_x.max():.2f}")
print(f"current y range: {current_y.min():.2f} to {current_y.max():.2f}")

In [ ]:
# Expand hotstart_500m_10layer to the full hydro_500m_origin x/y grid
# New cells introduced by this reindex are filled with -9999
hotstart_500m_10layer = hotstart_500m_10layer.reindex(
    xax=xc_500_target,
    yax=yc_500_target,
    fill_value=-9999
)

print("Padded Hotstart 500m dimensions:", dict(hotstart_500m_10layer.dims))
print(
    f"Padded x range: {hotstart_500m_10layer.xax.values.min():.2f} "
    f"to {hotstart_500m_10layer.xax.values.max():.2f}"
)
print(
    f"Padded y range: {hotstart_500m_10layer.yax.values.min():.2f} "
    f"to {hotstart_500m_10layer.yax.values.max():.2f}"
)

print(
    f"Added columns: {len(xc_500_target) - len(current_x)}, "
    f"Added rows: {len(yc_500_target) - len(current_y)}"
)

In [ ]:


hotstart_500m = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_201501_addrow.in')

# Define hydrological variables from hydro_500m (if available)
if 'hydro_500m' in globals():
    hydro_var_names = set(hydro_500m.data_vars)
else:
    try:
        hydro_ref = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_hydro_201501.in')
        hydro_var_names = set(hydro_ref.data_vars)
    except Exception:
        hydro_var_names = set()
        print("Warning: hydro_500m not available; treating all variables as biological.")

bio_var_names = [v for v in hotstart_500m.data_vars if v not in hydro_var_names]

print(f"Total variables in hotstart_500m: {len(hotstart_500m.data_vars)}")
print(f"Hydrological variables excluded: {len(hydro_var_names)}")
print(f"Biological variables to check: {len(bio_var_names)}\n")

summary = []
for var_name in bio_var_names:
    da = hotstart_500m[var_name]
    values = da.values

    # Count NaN as missing; also count explicit fill/missing flags when present
    missing_mask = np.isnan(values)

    fill_value = da.attrs.get('_FillValue', None)
    if fill_value is not None:
        missing_mask |= (values == fill_value)

    missing_value = da.attrs.get('missing_value', None)
    if missing_value is not None:
        missing_mask |= (values == missing_value)

    total = values.size
    missing = int(np.sum(missing_mask))
    pct = (missing / total * 100.0) if total > 0 else np.nan
    summary.append((var_name, missing, total, pct))

summary.sort(key=lambda x: x[3], reverse=True)

print("Biological variable missing-value summary:")
print(f"{'Variable':<28} {'Missing':>12} {'Total':>12} {'Missing %':>10}")
print("-" * 66)
for var_name, missing, total, pct in summary:
    print(f"{var_name:<28} {missing:>12,d} {total:>12,d} {pct:>9.2f}%")

Saved to '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_2015_01.in'


In [ ]:
# Create a test restart file where biological variables are set to 0.1
# while hydrological variables are kept unchanged.

test_restart = hotstart_500m.copy(deep=True)

# Determine hydrological variable names
if 'hydro_var_names' in globals() and isinstance(hydro_var_names, set):
    hydro_names = hydro_var_names
elif 'hydro_500m' in globals():
    hydro_names = set(hydro_500m.data_vars)
else:
    try:
        hydro_ref = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_hydro_201501.in')
        hydro_names = set(hydro_ref.data_vars)
    except Exception:
        hydro_names = set()
        print('Warning: could not determine hydrological variables; all variables treated as biological.')

bio_names = [v for v in test_restart.data_vars if v not in hydro_names]

updated = 0
skipped = 0
for var_name in bio_names:
    da = test_restart[var_name]

    # Only overwrite numeric variables
    if np.issubdtype(da.dtype, np.number):
        test_restart[var_name] = xr.full_like(da, 100, dtype=np.float32)
        updated += 1
    else:
        print(f'Skipping non-numeric variable: {var_name} (dtype={da.dtype})')
        skipped += 1

output_path = '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_test.in'
test_restart.to_netcdf(output_path)

print(f'Biological variables found: {len(bio_names)}')
print(f'Biological variables overwritten with 0.1: {updated}')
print(f'Biological variables skipped (non-numeric): {skipped}')
print(f'Saved test restart file to: {output_path}')